In [8]:
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [9]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 260
BATCH_SIZE = 16
ABLATION_EPOCHS = 20          # updated from 10 -> matches current tuned training budget
NUM_CLASSES = 3
NUM_WORKERS = 0
SEED = 42

CLASS_NAMES = ["BACTERIA", "NORMAL", "VIRUS"]

PROJECT_ROOT = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI")
DATA_ROOT = PROJECT_ROOT / "dataset" / "processed_dataset"
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR   = DATA_ROOT / "validation"
ABLATION_RESULTS_PATH = PROJECT_ROOT / "results" / "ablation_results_20ep.csv"
ABLATION_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [10]:
# ============================================================
# Cell 3: Data Preparation
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.90, 1.10)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=val_transform)

assert train_dataset.classes == CLASS_NAMES, f"Class order mismatch: {train_dataset.classes}"

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

train_labels = [label for _, label in train_dataset.samples]
class_weights = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_CLASSES), y=train_labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("Train samples:", len(train_dataset), "| Val samples:", len(val_dataset))
print("Class weights:", class_weights)

Train samples: 4099 | Val samples: 878
Class weights: [0.70212402 1.23315283 1.30749601]


In [11]:
# ============================================================
# Cell 4: Building Blocks (same definitions as Notebook 09)
# ============================================================

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction_ratio, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction_ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        return x * self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction_ratio)
        self.spatial_attention = SpatialAttention()
    def forward(self, x):
        return self.spatial_attention(self.channel_attention(x))

class MultiScaleFeatureFusion(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        mid = in_channels // reduction
        self.reduce = nn.Sequential(nn.Conv2d(in_channels, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.branch_1x1 = nn.Sequential(nn.Conv2d(mid, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.branch_3x3 = nn.Sequential(nn.Conv2d(mid, mid, 3, padding=1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.branch_5x5 = nn.Sequential(nn.Conv2d(mid, mid, 3, padding=2, dilation=2, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.fusion = nn.Sequential(nn.Conv2d(mid * 3, in_channels, 1, bias=False), nn.BatchNorm2d(in_channels), nn.ReLU(inplace=True))
    def forward(self, x):
        x = self.reduce(x)
        f = torch.cat([self.branch_1x1(x), self.branch_3x3(x), self.branch_5x5(x)], dim=1)
        return self.fusion(f)

class AdaptiveFeatureFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.weight_generator = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1, bias=False), nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 1, bias=False), nn.Sigmoid()
        )
    def forward(self, feature_a, feature_b):
        weights = self.weight_generator(torch.cat([feature_a, feature_b], dim=1))
        return weights * feature_a + (1.0 - weights) * feature_b

class ResidualEnhancement(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.refine = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1, bias=False), nn.BatchNorm2d(channels), nn.ReLU(inplace=True))
    def forward(self, x):
        return x + self.refine(x)

In [12]:
# ============================================================
# Cell 5: Configurable PneumoXNet for Ablation
# ============================================================
class PneumoXNetAblation(nn.Module):
    def __init__(self, num_classes=3, use_cbam=True, use_multiscale=True, use_aff=True, use_residual=True):
        super().__init__()
        self.use_cbam = use_cbam
        self.use_multiscale = use_multiscale
        self.use_aff = use_aff
        self.use_residual = use_residual

        weights = EfficientNet_B2_Weights.DEFAULT
        backbone = efficientnet_b2(weights=weights)
        self.backbone = backbone.features
        self.feature_channels = 1408

        if use_cbam:
            self.cbam = CBAM(self.feature_channels)
        if use_multiscale:
            self.multiscale = MultiScaleFeatureFusion(self.feature_channels)
        if use_aff:
            self.aff = AdaptiveFeatureFusion(self.feature_channels)
        if use_residual:
            self.residual = ResidualEnhancement(self.feature_channels)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.55),
            nn.Linear(self.feature_channels, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.45),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)

        branch_a = self.cbam(features) if self.use_cbam else features
        branch_b = self.multiscale(features) if self.use_multiscale else features

        if self.use_aff:
            fused = self.aff(branch_a, branch_b)
        else:
            fused = 0.5 * branch_a + 0.5 * branch_b   # uniform-average fallback, as in the paper

        enhanced = self.residual(fused) if self.use_residual else fused

        pooled = self.pool(enhanced)
        return self.classifier(pooled)

In [13]:
# ============================================================
# Cell 6: Train / Validate Functions (lightweight, no scheduler)
# ============================================================

def build_optimizer(model):
    backbone_params, new_module_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("backbone."):
            backbone_params.append(p)
        else:
            new_module_params.append(p)
    return optim.AdamW([
        {"params": backbone_params,    "lr": 7e-5, "weight_decay": 2e-4},
        {"params": new_module_params,  "lr": 1e-4, "weight_decay": 5e-4},
    ])

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss, all_preds, all_labels = 0.0, [], []
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    _, _, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    return running_loss / len(dataloader.dataset), acc, f1

def run_config(name, config_kwargs):
    set_seed(SEED)
    model = PneumoXNetAblation(num_classes=NUM_CLASSES, **config_kwargs).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)
    optimizer = build_optimizer(model)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=4)

    best_acc, best_f1 = 0.0, 0.0
    start = time.time()
    for epoch in range(ABLATION_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc, val_f1 = validate(model, val_loader, criterion, DEVICE)
        scheduler.step(val_loss)
        if val_acc > best_acc:
            best_acc, best_f1 = val_acc, val_f1
        print(f"[{name}] Epoch {epoch+1}/{ABLATION_EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")

    n_params = sum(p.numel() for p in model.parameters())
    elapsed = time.time() - start
    del model
    torch.cuda.empty_cache()
    return {
        "Configuration": name,
        "Valid_Accuracy": round(best_acc * 100, 2),
        "Valid_Macro_F1": round(best_f1, 4),
        "Params": n_params,
        "Time_sec": round(elapsed, 1),
    }

In [ ]:
# ============================================================
# Cell 7: Run Ablation Study
# ============================================================

ablation_configs = {
    "Full PneumoXNet":            dict(use_cbam=True,  use_multiscale=True,  use_aff=True,  use_residual=True),
    "w/o CBAM":                   dict(use_cbam=False, use_multiscale=True,  use_aff=True,  use_residual=True),
    "w/o Multi-Scale Fusion":     dict(use_cbam=True,  use_multiscale=False, use_aff=True,  use_residual=True),
    "w/o Adaptive Feature Fusion":dict(use_cbam=True,  use_multiscale=True,  use_aff=False, use_residual=True),
    "w/o Residual Enhancement":   dict(use_cbam=True,  use_multiscale=True,  use_aff=True,  use_residual=False),
}

ablation_results = []
for name, cfg in ablation_configs.items():
    result = run_config(name, cfg)
    ablation_results.append(result)

[Full PneumoXNet] Epoch 1/20 | train_loss=0.7766 train_acc=0.6999 | val_loss=0.6960 val_acc=0.7916 val_f1=0.7916
[Full PneumoXNet] Epoch 2/20 | train_loss=0.6764 train_acc=0.7817 | val_loss=0.7150 val_acc=0.7517 val_f1=0.7591
[Full PneumoXNet] Epoch 3/20 | train_loss=0.6580 train_acc=0.7931 | val_loss=0.6608 val_acc=0.8052 val_f1=0.8067
[Full PneumoXNet] Epoch 4/20 | train_loss=0.6427 train_acc=0.8121 | val_loss=0.7142 val_acc=0.7677 val_f1=0.7787
[Full PneumoXNet] Epoch 5/20 | train_loss=0.6160 train_acc=0.8197 | val_loss=0.7035 val_acc=0.7563 val_f1=0.7715
[Full PneumoXNet] Epoch 6/20 | train_loss=0.5962 train_acc=0.8322 | val_loss=0.6943 val_acc=0.7620 val_f1=0.7749
[Full PneumoXNet] Epoch 7/20 | train_loss=0.5707 train_acc=0.8497 | val_loss=0.6719 val_acc=0.7882 val_f1=0.7981
[Full PneumoXNet] Epoch 8/20 | train_loss=0.5597 train_acc=0.8575 | val_loss=0.7271 val_acc=0.7654 val_f1=0.7783
[Full PneumoXNet] Epoch 9/20 | train_loss=0.5269 train_acc=0.8763 | val_loss=0.6847 val_acc=0.78